In [0]:
# DAY1
#PySpark code to find duplicate records with dummy data

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count

spark = SparkSession.builder.appName("FindDuplicates").getOrCreate()

data = [
    (1, "Ravi", "IT", 50000),
    (2, "Anu", "HR", 60000),
    (3, "Ravi", "IT", 50000),
    (4, "Meera", "Finance", 70000),
    (5, "Anu", "HR", 60000)
]

columns = ["emp_id", "emp_name", "department", "salary"]
df = spark.createDataFrame(data, columns)
df.show()
df_duplicate = df.groupBy("emp_name", "department", "salary") \
    .count() \
    .filter(col("count") > 1)
df_duplicate.show()

#To display full duplicate rows
df_duplicate_rows =( 
    df.join(
        df_duplicate.select("emp_name", "department", "salary"), 
        on=["emp_name", "department", "salary"],
        how="inner"
    )
)
df_duplicate_rows.show()

# Remove the all the duplicate rows
df_no_duplicates = df.dropDuplicates(["emp_name", "department", "salary"])
df_no_duplicates.show()

# Another method
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
window_spec =(
     Window.partitionBy("emp_name", "department", "salary").orderBy(col("emp_id"))
)
df_no_duplicates = (
     df.withColumn("row_number", row_number().over(window_spec))
     .filter(col("row_number") == 1)
     .drop(col("row_number"))
)
df_no_duplicates.show()